In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, mean_absolute_error, r2_score
)
from sklearn.metrics import mean_squared_error
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model

import warnings
warnings.filterwarnings("ignore")

import shap

In [3]:
df = pd.read_excel("/content/drive/MyDrive/colab_datasets/elevator_fault_dataset_enhanced.xlsx")
df.head()

,Timestamp,Motor_Current,Motor_Voltage,Vibration_Level,Speed,Cabin_Position,Cabin_Acceleration,Cabin_Deceleration,Door_Operation_Time,Braking_Force,...,Health_Indices,RUL,Fault_State,Fault_Category,Fault_Severity,Temp_Category,Component_ID_Proxy,High_Speed_Hours,Load_Accumulation,Pulley_Wear_Accumulated
0,2020-01-01 00:00:00,22.289903,241.253222,1.393233,3.519079,19,0.349191,0.358643,5.747832,112.280046,...,171.975594,893.941155,1,Door Failure,Minor,Low (<= 40),1,0,845.186023,84.152230
1,2020-01-01 01:00:00,25.240487,236.078046,0.090303,2.226921,9,0.260127,0.266565,7.552277,142.770141,...,150.250886,2130.455491,0,Brake Failure,Critical,Medium (40-80),1,0,1923.604329,130.980306
2,2020-01-01 02:00:00,32.769471,432.220975,0.932987,2.523211,20,0.291057,0.161099,7.703547,98.016876,...,164.327588,2472.445475,0,Motor Malfunction,Minor,Low (<= 40),1,0,4278.648211,189.880795
3,2020-01-01 03:00:00,34.913962,645.108627,0.490413,4.910428,20,0.439094,0.214058,8.335525,75.588623,...,142.256458,1753.632090,0,Motor Malfunction,Moderate,Medium (40-80),2,0,785.277646,65.480344
4,2020-01-01 04:00:00,27.799073,224.451891,0.844176,5.188226,20,0.566116,0.365294,4.717031,95.882924,...,89.765157,2399.361778,0,Sensor Error,Critical,Medium (40-80),2,0,2796.725017,143.115920


In [4]:
df.columns

Index(['Timestamp', 'Motor_Current', 'Motor_Voltage', 'Vibration_Level',
       'Speed', 'Cabin_Position', 'Cabin_Acceleration', 'Cabin_Deceleration',
       'Door_Operation_Time', 'Braking_Force', 'Rope_Tension',
       'Guide_Rail_Condition_Index', 'Pulley_Wear', 'Lubrication_Levels',
       'Power_Consumption', 'Battery_Health', 'Voltage_Fluctuations',
       'Current_Harmonics', 'Ambient_Temperature', 'Humidity', 'Dust_Levels',
       'External_Vibration_Levels', 'Load', 'Passenger_Count',
       'Number_of_Trips', 'Stop_Frequency', 'Travel_Distance',
       'Call_Waiting_Time', 'Fault_Logs', 'Maintenance_History',
       'Component_Age', 'Last_Serviced_Days_Ago', 'MTTF', 'Signal_Noise',
       'Temperature_Gradient', 'Door_Sensor_Readings', 'Control_System_Delay',
       'Frequency_of_Emergency_Stops', 'Button_Press_Logs',
       'Door_Obstruction_Events', 'Overload_Events', 'Health_Indices', 'RUL',
       'Fault_State', 'Fault_Category', 'Fault_Severity', 'Temp_Category',
       

In [5]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [6]:
df = df.sort_values(['Component_ID_Proxy', 'Timestamp'])

In [ ]:
degradation_features = [
    'Motor_Current', 'Vibration_Level', 'Pulley_Wear',
    'Temperature_Gradient', 'Load', 'Power_Consumption'
]

agg_df = (
    df.groupby('Component_ID_Proxy')
    .agg({
        **{col: ['mean', 'std', 'last'] for col in degradation_features},
        'High_Speed_Hours': 'max',
        'Load_Accumulation': 'max',
        'Pulley_Wear_Accumulated': 'max',
        'RUL': 'last',
        'Fault_State': 'last',
        'Fault_Severity': 'last',
        'MTTF': 'last'
    })
)

agg_df.columns = ['_'.join(col) for col in agg_df.columns]
agg_df.reset_index(inplace=True)

In [ ]:
le_fault = LabelEncoder()
agg_df['Fault_State_last'] = le_fault.fit_transform(
    agg_df['Fault_State_last']
)

In [ ]:
X = agg_df.drop([
    'Component_ID_Proxy',
    'RUL_last',
    'Fault_State_last',
    'Fault_Severity_last',
    'MTTF_last'
], axis=1)

y_fault = agg_df['Fault_State_last']
y_rul = agg_df['RUL_last']

groups = agg_df['Component_ID_Proxy']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
gkf = GroupKFold(n_splits=5)

fault_scores = []
rul_scores = []

for train_idx, test_idx in gkf.split(X_scaled, y_fault, groups):

    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_fault_train, y_fault_test = y_fault.iloc[train_idx], y_fault.iloc[test_idx]
    y_rul_train, y_rul_test = y_rul.iloc[train_idx], y_rul.iloc[test_idx]

    # ---- Fault Model ----
    clf = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss'
    )
    clf.fit(X_train, y_fault_train)
    y_pred_fault = clf.predict(X_test)

    fault_scores.append(f1_score(y_fault_test, y_pred_fault, average='weighted'))

    # ---- RUL Model ----
    reg = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05
    )
    reg.fit(X_train, y_rul_train)
    y_pred_rul = reg.predict(X_test)

    rul_scores.append(mean_absolute_error(y_rul_test, y_pred_rul))

print("Fault F1 (CV):", np.mean(fault_scores))
print("RUL MAE (CV):", np.mean(rul_scores))

In [ ]:
y_surv = Surv.from_arrays(
    event=(agg_df['RUL_last'] <= 0),
    time=agg_df['MTTF_last']
)

rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=15,
    n_jobs=-1
)
rsf.fit(X_scaled, y_surv)


In [ ]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_scaled)

shap.summary_plot(shap_values, X, show=False)
plt.show()

New pipeline

In [16]:
TARGET_FAULT = "Fault_State"
TARGET_RUL = "RUL"
TARGET_SEVERITY = "Fault_Severity"
FAULT_CATEGORY = "Fault_Category"
TEMP_CATEGORY = "Temp_Category"

DROP_COLS = [
    "Timestamp",
    "RUL_Stage_Label",
    "RUL_Stage",
    "Component_ID_Proxy",
    TARGET_FAULT,
    TARGET_RUL,
    TARGET_SEVERITY,
    FAULT_CATEGORY,
    TEMP_CATEGORY
]

In [9]:
severity_encoder = LabelEncoder()
df[TARGET_SEVERITY] = severity_encoder.fit_transform(df[TARGET_SEVERITY])

In [17]:
X = df.drop(columns=DROP_COLS)
y_fault = df[TARGET_FAULT]
y_rul = df[TARGET_RUL]

In [13]:
X.columns

Index(['Motor_Current', 'Motor_Voltage', 'Vibration_Level', 'Speed',
       'Cabin_Position', 'Cabin_Acceleration', 'Cabin_Deceleration',
       'Door_Operation_Time', 'Braking_Force', 'Rope_Tension',
       'Guide_Rail_Condition_Index', 'Pulley_Wear', 'Lubrication_Levels',
       'Power_Consumption', 'Battery_Health', 'Voltage_Fluctuations',
       'Current_Harmonics', 'Ambient_Temperature', 'Humidity', 'Dust_Levels',
       'External_Vibration_Levels', 'Load', 'Passenger_Count',
       'Number_of_Trips', 'Stop_Frequency', 'Travel_Distance',
       'Call_Waiting_Time', 'Fault_Logs', 'Maintenance_History',
       'Component_Age', 'Last_Serviced_Days_Ago', 'MTTF', 'Signal_Noise',
       'Temperature_Gradient', 'Door_Sensor_Readings', 'Control_System_Delay',
       'Frequency_of_Emergency_Stops', 'Button_Press_Logs',
       'Door_Obstruction_Events', 'Overload_Events', 'Health_Indices',
       'Component_ID_Proxy', 'High_Speed_Hours', 'Load_Accumulation',
       'Pulley_Wear_Accumulated'

In [18]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [34]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_scaled, y_fault,
    test_size=0.2,
    stratify=y_fault,
    random_state=42
)

In [35]:
classes = np.unique(y_train_f)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_f
)

class_weight_dict = dict(zip(classes, class_weights))
print(class_weight_dict)

{np.int64(0): np.float64(0.525766590389016), np.int64(1): np.float64(10.202486678507993)}


In [36]:
smote = SMOTE(random_state=42)
X_train_f, y_train_f = smote.fit_resample(X_train_f, y_train_f)

In [38]:
input_f = Input(shape=(X_train_f.shape[1],))

x = Dense(256, activation="relu")(input_f)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.2)(x)
x = Dense(64, activation="relu")(x)

output_f = Dense(1, activation="sigmoid")(x)

fault_model = Model(inputs=input_f, outputs=output_f)

fault_model.compile(
    optimizer=Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fault_model.fit(
    X_train_f,
    y_train_f,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.3729 - loss: 1.1846 - val_accuracy: 0.9991 - val_loss: 0.0696
Epoch 2/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.4454 - loss: 0.8899 - val_accuracy: 0.9973 - val_loss: 0.0474
Epoch 3/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5454 - loss: 0.7618 - val_accuracy: 0.9969 - val_loss: 0.0402
Epoch 4/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.6327 - loss: 0.6552 - val_accuracy: 0.9972 - val_loss: 0.0336
Epoch 5/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.6767 - loss: 0.5986 - val_accuracy: 0.9985 - val_loss: 0.0293
Epoch 6/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7158 - loss: 0.5426 - val_accuracy: 0.9986 - val_loss: 0.0290
Epoch 7/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7401 - loss: 0.5027 - val_accuracy: 0.9982 - val_loss: 0.0281
Epoch 8/50
1639/1639 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7663 - loss: 0.4711

In [46]:
y_pred_f = (fault_model.predict(X_test_f) > 0.2).astype(int)

print("\nFAULT CLASSIFICATION PERFORMANCE")
print("Accuracy:", accuracy_score(y_test_f, y_pred_f))
print(classification_report(y_test_f, y_pred_f))
print("Confusion Matrix:\n", confusion_matrix(y_test_f, y_pred_f))

270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

FAULT CLASSIFICATION PERFORMANCE
Accuracy: 0.5983520946965302
              precision    recall  f1-score   support

           0       0.96      0.61      0.74      8195
           1       0.06      0.46      0.10       422

    accuracy                           0.60      8617
   macro avg       0.51      0.53      0.42      8617
weighted avg       0.91      0.60      0.71      8617

Confusion Matrix:
 [[4963 3232]
 [ 229  193]]


In [45]:
y_prob = fault_model.predict(X_test_f).ravel()
t = 0.2
y_pred = (y_prob >= t).astype(int)
print(f"\nThreshold = {t}")
print(classification_report(y_test_f, y_pred))

270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Threshold = 0.2
              precision    recall  f1-score   support

           0       0.96      0.61      0.74      8195
           1       0.06      0.46      0.10       422

    accuracy                           0.60      8617
   macro avg       0.51      0.53      0.42      8617
weighted avg       0.91      0.60      0.71      8617



In [23]:
y_rul_log = np.log1p(y_rul)

In [24]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_scaled,
    y_rul_log,
    test_size=0.2,
    random_state=42
)

In [25]:
input_r = Input(shape=(X_train_r.shape[1],))

x = Dense(256, activation="relu")(input_r)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.2)(x)
x = Dense(64, activation="relu")(x)

output_r = Dense(1, activation="linear")(x)

rul_model = Model(inputs=input_r, outputs=output_r)

rul_model.compile(
    optimizer=Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

rul_model.fit(
    X_train_r,
    y_train_r,
    validation_split=0.2,
    epochs=80,
    batch_size=32,
    verbose=1
)

Epoch 1/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 4.2098 - mae: 1.3363 - val_loss: 0.7725 - val_mae: 0.7380
Epoch 2/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.6375 - mae: 0.6397 - val_loss: 0.4352 - val_mae: 0.5561
Epoch 3/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.3748 - mae: 0.4934 - val_loss: 0.3415 - val_mae: 0.5019
Epoch 4/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.2531 - mae: 0.4079 - val_loss: 0.2799 - val_mae: 0.4580
Epoch 5/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.2048 - mae: 0.3694 - val_loss: 0.2349 - val_mae: 0.4198
Epoch 6/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.1863 - mae: 0.3532 - val_loss: 0.1806 - val_mae: 0.3534
Epoch 7/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.1791 - mae: 0.3469 - val_loss: 0.1795 - val_mae: 0.3483
Epoch 8/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1775 - mae: 0.3457 - val_loss: 0.1790 - val_mae: 0.3470
Epoch 9/80
862/862 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - lo

In [26]:
# Predict in log space
y_pred_log = rul_model.predict(X_test_r).ravel()

# Safety: prevent negative log predictions
y_pred_log = np.clip(y_pred_log, a_min=0, a_max=None)

# Inverse log transform
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_r)

# Safety: prevent negative RUL
y_pred = np.clip(y_pred, a_min=0, a_max=None)
y_true = np.clip(y_true, a_min=0, a_max=None)

print("\nDEEP RUL REGRESSION PERFORMANCE")
print("MAE :", mean_absolute_error(y_true, y_pred))
print("R²  :", r2_score(y_true, y_pred))
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print("RMSE:", rmse)

270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

DEEP RUL REGRESSION PERFORMANCE
MAE : 636.7308723851248
R²  : -0.05123016910552769
RMSE: 775.4866686011461


Treating RUL Regression as Classification by converting the RUL into HEALTH STAGE CLASSIFICATION

In [ ]:
# Create RUL Health Stage
def rul_stage(rul):
    if rul > 2200:
        return "Healthy"
    elif rul > 1610:
        return "Degrading"
    else:
        return "Critical"

df["RUL_Stage"] = df["RUL"].apply(rul_stage)

# Encode labels
from sklearn.preprocessing import LabelEncoder
le_stage = LabelEncoder()
df["RUL_Stage_Label"] = le_stage.fit_transform(df["RUL_Stage"])

In [ ]:
drop_cols = [
    "Timestamp",
    "RUL",
    "RUL_Stage",
    "RUL_Stage_Label",
    "Fault_Category",
    "Fault_Severity",
    "Component_ID_Proxy",
    "Temp_Category"
]

X = df.drop(columns=drop_cols)
y = df["RUL_Stage_Label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train))
X_test  = scaler.transform(imputer.transform(X_test))

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

In [ ]:
tf.random.set_seed(42)

inp = Input(shape=(X_train.shape[1],))

x = Dense(256, activation="relu")(inp)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(64, activation="relu")(x)

out = Dense(3, activation="softmax")(x)

model = Model(inp, out)

def sparse_categorical_focal_loss(gamma=2.0, alpha=None):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_onehot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        ce = -y_true_onehot * tf.math.log(y_pred)
        weight = tf.pow(1.0 - y_pred, gamma)

        if alpha is not None:
            alpha_factor = y_true_onehot * alpha
            weight *= alpha_factor

        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=1))
    return loss

loss_fn = sparse_categorical_focal_loss(gamma=2.0)

def ordinal_penalty(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred_class = tf.argmax(y_pred, axis=1)
    y_pred_class = tf.cast(y_pred_class, tf.float32)
    return tf.reduce_mean(tf.square(y_true - y_pred_class))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        label_smoothing=0.1
    ),
    metrics=["accuracy"]
)

es = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=80,
    batch_size=64,
    callbacks=[es],
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print("\nRUL HEALTH STAGE CLASSIFICATION REPORT\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=le_stage.classes_,
    digits=3
))

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

**-----------------------------------------------------------------**

In [7]:
# =========================================================
# 3. CREATE RUL HEALTH STAGE (UNCHANGED)
# =========================================================
def rul_stage(rul):
    if rul > 3000:
        return "Healthy"
    elif rul > 1500:
        return "Degrading"
    else:
        return "Critical"

df["RUL_Stage"] = df["RUL"].apply(rul_stage)

le_stage = LabelEncoder()
df["RUL_Stage_Label"] = le_stage.fit_transform(df["RUL_Stage"])


# =========================================================
# 4. FEATURE SELECTION (NO LEAKAGE)
# =========================================================
drop_cols = [
    "Timestamp",
    "RUL",
    "RUL_Stage",
    "RUL_Stage_Label",
    "Fault_Category",
    "Fault_Severity",
    "Component_ID_Proxy",
    "Temp_Category"
]

X = df.drop(columns=drop_cols)
y = df["RUL_Stage_Label"].values


# =========================================================
# 5. TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# =========================================================
# 6. PREPROCESSING
# =========================================================
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train))
X_test  = scaler.transform(imputer.transform(X_test))


# =========================================================
# 7. ONE-HOT LABELS (FOR LABEL SMOOTHING)
# =========================================================
num_classes = 3
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_oh  = tf.keras.utils.to_categorical(y_test, num_classes)


# =========================================================
# 8. CLASS WEIGHTS
# =========================================================
# class_weights = compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(y_train),
#     y=y_train
# )

# class_weight_dict = class_weight_dict = {
#     0: 1.8,  # Critical
#     1: 1.0,  # Degrading
#     2: 2.2   # Healthy (minority)
# }

class_weight_dict = {
    0: 1.9,   # Critical
    1: 0.75,  # Degrading (KEY CHANGE)
    2: 3.2    # Healthy
}

# =========================================================
# 9. DEEP NEURAL NETWORK (STABLE)
# =========================================================
tf.random.set_seed(42)

inp = Input(shape=(X_train.shape[1],))

x = Dense(256, activation="relu")(inp)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(64, activation="relu")(x)

out = Dense(3, activation="softmax")(x)

model = Model(inp, out)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=["accuracy"]
)

es = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


# =========================================================
# 10. TRAINING
# =========================================================
history = model.fit(
    X_train,
    y_train_oh,
    validation_split=0.2,
    epochs=80,
    batch_size=64,
    callbacks=[es],
    class_weight=class_weight_dict,
    verbose=1
)


# =========================================================
# 11. EVALUATION
# =========================================================
# y_pred = np.argmax(model.predict(X_test), axis=1)
logits = model.predict(X_test)
temperature = 1.6   # key hyperparameter
probs = tf.nn.softmax(logits / temperature, axis=1).numpy()
y_pred = np.argmax(probs, axis=1)

print("\nRUL HEALTH STAGE CLASSIFICATION REPORT\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=le_stage.classes_,
    digits=3
))

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

Epoch 1/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.3487 - loss: 1.6442 - val_accuracy: 0.3823 - val_loss: 1.0902
Epoch 2/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.3957 - loss: 1.4634 - val_accuracy: 0.3808 - val_loss: 1.0857
Epoch 3/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4064 - loss: 1.4348 - val_accuracy: 0.3704 - val_loss: 1.0813
Epoch 4/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4230 - loss: 1.4169 - val_accuracy: 0.3614 - val_loss: 1.0808
Epoch 5/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4239 - loss: 1.4043 - val_accuracy: 0.3592 - val_loss: 1.0809
Epoch 6/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.4214 - loss: 1.3981 - val_accuracy: 0.3637 - val_loss: 1.0794
Epoch 7/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.4355 - loss: 1.3919 - val_accuracy: 0.3515 - val_loss: 1.0834
Epoch 8/80
431/431 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.4328 - loss: 1.3783 - val_accuracy: 

In [27]:
fault_model.save("fault_model.keras")
model.save("rul_model.keras")

In [ ]:
plot_model(
    fault_model,
    to_file="fault_model_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=200
)

In [ ]:
plot_model(
    model,
    to_file="rul_model_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=200
)